#Vectorization

In [ ]:
# FFNN for Fake News Classification using TF-IDF

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# 1. Load dataset
df = pd.read_csv("WELFake_Dataset.csv", engine="python", on_bad_lines="skip")

# 2. Basic cleaning
df = df.dropna(subset=["title", "text", "label"])
*********************
df = df.dropna(subset=["label"])
df["label"] = df["label"].astype(int)
df = df[df["label"].isin([0, 1])]
*********************

# 3. Split features and target
X = df["content"]
y = df["label"]

# 4. Train, validation, test split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
*********************

# 5. Convert text into TF-IDF vectors
*********************
X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(X_test)

# 6. Build FFNN model
model = Sequential([
    Dense(128, activation="relu", input_shape=(X_train_tfidf.shape[1],)),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.3),
*********************
])

# 7. Compile model
*********************

# 8. Early stopping
early_stop = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

# 9. Train model
history = model.fit(X_train_tfidf, y_train, validation_data=(X_val_tfidf, y_val),
                    epochs=10, batch_size=64, callbacks=[early_stop], verbose=1)

# 10. Evaluate model
test_loss, test_accuracy = model.evaluate(X_test_tfidf, y_test, verbose=0)
print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

# 11. Predictions
y_pred_prob = model.predict(X_test_tfidf)
*********************

print("Accuracy Score:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
*********************

# 12. Example predictions
results = pd.DataFrame({"Actual Label": y_test.values, "Predicted Label": y_pred, "Real Probability": y_pred_prob.flatten()})
print(results.head(10))

#Word Embedding

In [ ]:
# Showing word vectors from Word2Vec, GloVe, and FastText

!pip install gensim
import gensim.downloader as api
import numpy as np

# 1. Choose a word
word = "computer"

# 2. Load pretrained embedding models
word2vec_model = api.load("word2vec-google-news-300")
*********************
fasttext_model = api.load("fasttext-wiki-news-subwords-300")

# 3. Get vectors
word2vec_vector = word2vec_model[word]
*********************
fasttext_vector = fasttext_model[word]

# 4. Print vector sizes
print("Word2Vec vector size:", word2vec_vector.shape)
print("GloVe vector size:", glove_vector.shape)
print("FastText vector size:", fasttext_vector.shape)

# 5. Print first 10 values from each vector
print("\nWord2Vec vector for:", word)
print(word2vec_vector[:10])

print("\nGloVe vector for:", word)
print(glove_vector[:10])

print("\nFastText vector for:", word)
print(fasttext_vector[:10])

In [ ]:
# FFNN for Fake News Classification using Word Embeddings
import pandas as pd
import numpy as np
import re
import gensim.downloader as api
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import tensorflow as tf
*********************
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# 1. Load dataset
df = pd.read_csv("WELFake_Dataset.csv", engine="python", on_bad_lines="skip")

# 2. Basic cleaning
df = df.dropna(subset=["title", "text", "label"])
df["label"] = pd.to_numeric(df["label"], errors="coerce")
df = df.dropna(subset=["label"])
df["label"] = df["label"].astype(int)
df = df[df["label"].isin([0, 1])]
df["content"] = df["title"].astype(str) + " " + df["text"].astype(str)

# 3. Split features and target
X = df["content"]
y = df["label"]

# 4. Train, validation, test split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

# 5. Choose and load word embedding model
embedding_choice = "glove"
# embedding_choice = "word2vec"
# embedding_choice = "fasttext"

if embedding_choice == "glove":
*********************
elif embedding_choice == "word2vec":
    embedding_model = api.load("word2vec-google-news-300")
elif embedding_choice == "fasttext":
    embedding_model = api.load("fasttext-wiki-news-subwords-300")

embedding_dim = embedding_model.vector_size
print("Embedding model:", embedding_choice)
print("Embedding dimension:", embedding_dim)

# 6. Tokenizer
def tokenize(text):
    text = text.lower()
*********************
    return text.split()

# 7. Convert each document into one average embedding vector
def document_to_vector(text):
    words = tokenize(text)
    vectors = [embedding_model[word] for word in words if word in embedding_model]
    if len(vectors) == 0:
        return np.zeros(embedding_dim)
*********************

X_train_embed = np.array([document_to_vector(text) for text in X_train])
*********************
X_test_embed = np.array([document_to_vector(text) for text in X_test])

# 8. Build FFNN model
model = Sequential([
    Dense(128, activation="relu", input_shape=(X_train_embed.shape[1],)),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

# 9. Compile model
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# 10. Early stopping
early_stop = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

# 11. Train model
history = model.fit(X_train_embed, y_train, validation_data=(X_val_embed, y_val), epochs=10, batch_size=64, callbacks=[early_stop], verbose=1)

# 12. Evaluate model
test_loss, test_accuracy = model.evaluate(X_test_embed, y_test, verbose=0)
print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

# 13. Predictions
*********************
y_pred = (y_pred_prob >= 0.5).astype(int).flatten()

print("Accuracy Score:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Fake", "Real"]))

# 14. Example predictions
results = pd.DataFrame({"Actual Label": y_test.values, "Predicted Label": y_pred, "Real Probability": y_pred_prob.flatten()})
print(results.head(10))